# Trace analysis

Load an eval JSONL (with `response` field) and inspect individual traces.

*Baseline right, openmath wrong*: 3, 33, 97, 182, 185, 289, 410, 480, 519, 565, 613, 705, 733, 767, 800, 826

*Baseline wrong, openmath right*: 111, 148, 226, 246, 291, 317, 402, 417, 949, 1035

In [1]:
import json
from pathlib import Path

FILE = Path("../data/openmath_eval.jsonl")

rows = [json.loads(l) for l in FILE.open()]
print(f"Loaded {len(rows)} rows from {FILE.name}")
print(f"Keys: {list(rows[0].keys())}")

Loaded 225 rows from openmath_eval.jsonl
Keys: ['id', 'is_mcq', 'gold', 'response', 'correct']


In [2]:
# quick stats
n_correct = sum(r['correct'] for r in rows if 'correct' in r)
n_mcq = sum(r['is_mcq'] for r in rows if 'is_mcq' in r)
n_ff = sum(not r['is_mcq'] for r in rows if 'is_mcq' in r)
print(f"Overall: {n_correct}/{len(rows)} ({n_correct/len(rows)*100:.1f}%)")
print(f"MCQ: {n_mcq}  Free-form: {n_ff}")
lens = [len(r['response']) for r in rows if 'response' in r]
print(f"Response length  mean={sum(lens)//len(lens)}  min={min(lens)}  max={max(lens)} chars")

Overall: 139/225 (61.8%)
MCQ: 75  Free-form: 150
Response length  mean=10134  min=798  max=51263 chars


In [3]:
# print trace by question id
QID = 182  # change this

r = next((x for x in rows if x['id'] == QID), None)
if r is None:
    print(f"id={QID} not found")
else:
    print(f"id={r['id']}  is_mcq={r.get('is_mcq')}  correct={r.get('correct')}")
    print(f"gold: {r.get('gold')}")
    print(f"response length: {len(r['response'])} chars")
    print('-' * 60)
    print(r['response'])

id=182  is_mcq=True  correct=False
gold: F
response length: 26909 chars
------------------------------------------------------------
Okay, let's try to figure out this problem. So, the question is about finding the output sequence y_list corresponding to the input x_list [29, 30, 31, 32, 33, 34, 35, 36, 37, 38]. The definition given is that a(n) is the largest order of the automorphism group of a tournament with n nodes. Hmm, tournaments... right, a tournament is a directed graph where between every pair of nodes, there's exactly one directed edge. So each pair has a single directed edge in one direction or the other.

Now, the automorphism group of a graph is the set of all permutations of the nodes that preserve the edges. The order of the automorphism group would be the number of such permutations. So, the problem is asking for the largest possible automorphism group size for a tournament with n nodes. Wait, but tournaments can have different structures, so maybe some tournaments ha

In [4]:
# browse wrong answers
wrong = [r for r in rows if not r.get('correct', True)]
print(f"{len(wrong)} wrong answers")

WRONG_IDX = 10  # change this
r = wrong[WRONG_IDX]
print(f"\nid={r['id']}  is_mcq={r.get('is_mcq')}")
print(f"gold: {r.get('gold')}")
print(f"response length: {len(r['response'])} chars")
print('-' * 60)
print(r['response'])

86 wrong answers

id=127  is_mcq=False
gold: ['172.680555555555', '174.228888888889', '7.17799319321462', '6.7688366501569', '0.592589807558812', '77.7330455545177', '0.558811286099472', '179.436699624622', '0.669035334049935', '1.973381', '178.116438008079', '180.756961241164', '175.033660673377', '180.058005993289', 'information']
response length: 22411 chars
------------------------------------------------------------
Okay, let me try to work through this problem step by step. First, I need to compute the summary statistics for parts a and b. Then parts c, d, and e. Since this is a lot, I should start with part a.

Part a asks for the sample means of x and y, the standard deviations of x and y, and the correlation coefficient. The data is given as R vectors. Since I can't run R here, I'll have to calculate these manually or use some formulas. Wait, but maybe I can approximate or recall the formulas. Hmm, but with 100 data points (I count the vectors: x has 100 elements, y has 100 el

In [5]:
import json
import re
from pathlib import Path

REGRESSION_IDS = {3, 33, 97, 182, 185, 289, 410, 480, 519, 565, 613, 705, 733, 767, 800, 826}
GAIN_IDS       = {111, 148, 226, 246, 291, 317, 402, 417, 949, 1035}
KEYWORDS       = re.compile(r'\b(hard|difficult|challenging|tricky)\b', re.IGNORECASE)

def load(path):
    return {r['id']: r for r in (json.loads(l) for l in Path(path).open())}

baseline = load('../data/baseline_eval.jsonl')
openmath = load('../data/openmath_eval.jsonl')
shared   = set(baseline) & set(openmath)

def keyword_hits(rows_dict, ids):
    ids = ids & set(rows_dict)
    hits = sum(bool(KEYWORDS.search(rows_dict[i]['response'])) for i in ids)
    return hits, len(ids)

groups = [
    ('All shared',                         shared),
    ('Regressions (baseline✓ openmath✗)', REGRESSION_IDS),
    ('Gains       (baseline✗ openmath✓)', GAIN_IDS),
]

print(f"{'Group':<40} {'Baseline':>12} {'Openmath':>12}")
print('-' * 66)
for label, ids in groups:
    bh, bn = keyword_hits(baseline, ids)
    oh, on = keyword_hits(openmath, ids)
    print(f"{label:<40} {bh}/{bn} ({bh/bn*100:4.1f}%)   {oh}/{on} ({oh/on*100:4.1f}%)")


Group                                        Baseline     Openmath
------------------------------------------------------------------
All shared                               32/225 (14.2%)   20/225 ( 8.9%)
Regressions (baseline✓ openmath✗)        3/16 (18.8%)   5/16 (31.2%)
Gains       (baseline✗ openmath✓)        7/10 (70.0%)   2/10 (20.0%)


In [6]:
# Trace length: openmath wrong vs baseline wrong
import json, statistics
from pathlib import Path

baseline = {r['id']: r for r in (json.loads(l) for l in Path('../data/baseline_eval.jsonl').open())}
openmath = {r['id']: r for r in (json.loads(l) for l in Path('../data/openmath_eval.jsonl').open())}

base_wrong  = [r for r in baseline.values() if not r.get('correct')]
open_wrong  = [r for r in openmath.values() if not r.get('correct')]
base_right  = [r for r in baseline.values() if r.get('correct')]
open_right  = [r for r in openmath.values() if r.get('correct')]

def stats(rows):
    lens = [len(r['response']) for r in rows]
    return len(lens), statistics.mean(lens), statistics.median(lens), min(lens), max(lens)

groups = [
    ('Baseline wrong', base_wrong),
    ('Baseline right', base_right),
    ('Openmath wrong', open_wrong),
    ('Openmath right', open_right),
]

print(f"{'Group':<20} {'n':>4}  {'mean':>7}  {'median':>7}  {'min':>6}  {'max':>7}  chars")
print('-' * 65)
for label, rows in groups:
    n, mean, med, lo, hi = stats(rows)
    print(f"{label:<20} {n:>4}  {mean:>7,.0f}  {med:>7,.0f}  {lo:>6,}  {hi:>7,}")


Group                   n     mean   median     min      max  chars
-----------------------------------------------------------------
Baseline wrong         80   19,033   17,594   1,170   50,190
Baseline right        145   11,416    7,011   1,467   46,934
Openmath wrong         86   13,191   10,487     798   51,263
Openmath right        139    8,244    4,663   1,441   38,188


In [7]:
# For wrong answers: how many have the boxed answer only inside <think>, not in the final section?
import json, re
from pathlib import Path

BOXED   = re.compile(r'\\boxed\{', re.IGNORECASE)
MCQ_ANS = re.compile(r'\b[A-E]\b')  # MCQ letter in final section

def load(path):
    return [json.loads(l) for l in Path(path).open()]

def split_think(response):
    if '</think>' in response:
        idx = response.index('</think>')
        return response[:idx], response[idx + len('</think>'):]
    return response, ''

def answer_only_in_think(r):
    think, final = split_think(r['response'])
    has_boxed_in_think = bool(BOXED.search(think))
    has_boxed_in_final = bool(BOXED.search(final))
    # For MCQ with no boxed at all, check for a bare letter in final
    has_letter_in_final = bool(MCQ_ANS.search(final.strip()))
    answer_present_in_final = has_boxed_in_final or (r.get('is_mcq') and has_letter_in_final)
    return has_boxed_in_think and not answer_present_in_final

baseline = load('../data/baseline_eval.jsonl')
openmath = load('../data/openmath_eval.jsonl')

for name, rows in [('Baseline', baseline), ('Openmath', openmath)]:
    wrong = [r for r in rows if not r.get('correct')]
    embedded = [r for r in wrong if answer_only_in_think(r)]
    print(f"{name} wrong: {len(wrong)}  answer only in think: {len(embedded)} ({len(embedded)/len(wrong)*100:.1f}%)")
    if embedded:
        print(f"  sample IDs: {[r['id'] for r in embedded[:8]]}")


Baseline wrong: 80  answer only in think: 1 (1.2%)
  sample IDs: [568]
Openmath wrong: 86  answer only in think: 0 (0.0%)


# sft-005 vs pub-002 on `geometry_dev` (133 rows: 44 MCQ + 89 free-form)

Goal: locate **where** the geo MCQ +6.82 pp lift comes from and **why** free-form drops −1.13 pp.

- **pub-002 baseline:** `data/full_public_16k.jsonl` filtered to `watch_manifest.json` geometry ids.
- **sft-005 eval:** `data/openmath_geo_eval.jsonl` — download from Drive `results/sft_eval/openmath_geo_1k/eval_0.jsonl`.

Split every comparison by `is_mcq` so the MCQ signal and free-form signal don't smear.

In [8]:
# Load pub-002 (geo slice) + sft-005 (geometry_dev eval); build regression / gain id sets split by MCQ vs free-form
import json
from pathlib import Path

REPO = Path('..')
GEO_IDS = set(json.load(open(REPO / 'data/eval/watch_manifest.json'))['watch']['geometry']['ids'])

pub002_all = [json.loads(l) for l in (REPO / 'data/full_public_16k.jsonl').open()]
pub002 = {r['id']: r for r in pub002_all if r['id'] in GEO_IDS}

SFT005_PATH = REPO / 'data/openmath_geo_eval.jsonl'
sft005 = {r['id']: r for r in (json.loads(l) for l in SFT005_PATH.open())}

shared = set(pub002) & set(sft005)
assert shared, f"No id overlap. pub002 geo={len(pub002)}, sft005={len(sft005)}"

mcq_ids = {i for i in shared if pub002[i]['is_mcq']}
ff_ids  = {i for i in shared if not pub002[i]['is_mcq']}

def split(ids):
    regr = {i for i in ids if pub002[i]['correct'] and not sft005[i]['correct']}
    gain = {i for i in ids if not pub002[i]['correct'] and sft005[i]['correct']}
    both_right = {i for i in ids if pub002[i]['correct'] and sft005[i]['correct']}
    both_wrong = {i for i in ids if not pub002[i]['correct'] and not sft005[i]['correct']}
    return regr, gain, both_right, both_wrong

mcq_regr, mcq_gain, mcq_RR, mcq_WW = split(mcq_ids)
ff_regr,  ff_gain,  ff_RR,  ff_WW  = split(ff_ids)

print(f"Shared ids: {len(shared)}  (MCQ {len(mcq_ids)} / FF {len(ff_ids)})\n")
print(f"{'slice':<10} {'regr':>6} {'gain':>6} {'both✓':>6} {'both✗':>6}   net  Δacc")
print('-' * 56)
for label, regr, gain, RR, WW, n in [
    ('MCQ', mcq_regr, mcq_gain, mcq_RR, mcq_WW, len(mcq_ids)),
    ('FF',  ff_regr,  ff_gain,  ff_RR,  ff_WW,  len(ff_ids)),
]:
    net = len(gain) - len(regr)
    print(f"{label:<10} {len(regr):>6} {len(gain):>6} {len(RR):>6} {len(WW):>6}   {net:+4d}  {net/n*100:+5.2f} pp")

print("\nMCQ regressions: ", sorted(mcq_regr))
print("MCQ gains:       ", sorted(mcq_gain))
print("FF regressions:  ", sorted(ff_regr))
print("FF gains:        ", sorted(ff_gain))

Shared ids: 21  (MCQ 9 / FF 12)

slice        regr   gain  both✓  both✗   net  Δacc
--------------------------------------------------------
MCQ             0      2      6      1     +2  +22.22 pp
FF              0      0      4      8     +0  +0.00 pp

MCQ regressions:  []
MCQ gains:        [148, 294]
FF regressions:   []
FF gains:         []


In [9]:
# Trace length stats per group — does compression correlate with regression?
import statistics

def lens(rows_dict, ids):
    return [len(rows_dict[i]['response']) for i in ids if i in rows_dict]

def row(label, pub_l, sft_l):
    if not pub_l or not sft_l:
        return f"{label:<28} (empty)"
    dp = statistics.mean(pub_l)
    ds = statistics.mean(sft_l)
    pct = (ds - dp) / dp * 100 if dp else 0
    return (f"{label:<28} n={len(pub_l):>3}  "
            f"pub={dp:>7,.0f}  sft={ds:>7,.0f}  Δ={ds-dp:+7,.0f} ({pct:+5.1f}%)  "
            f"med pub={statistics.median(pub_l):>6,.0f} sft={statistics.median(sft_l):>6,.0f}")

groups = [
    ('MCQ all',          mcq_ids),
    ('MCQ regressions',  mcq_regr),
    ('MCQ gains',        mcq_gain),
    ('MCQ both right',   mcq_RR),
    ('MCQ both wrong',   mcq_WW),
    ('FF all',           ff_ids),
    ('FF regressions',   ff_regr),
    ('FF gains',         ff_gain),
    ('FF both right',    ff_RR),
    ('FF both wrong',    ff_WW),
]

print("Mean response length (chars) — pub-002 vs sft-005 per id group\n")
for label, ids in groups:
    print(row(label, lens(pub002, ids), lens(sft005, ids)))

Mean response length (chars) — pub-002 vs sft-005 per id group

MCQ all                      n=  9  pub= 24,721  sft= 20,896  Δ= -3,824 (-15.5%)  med pub=20,178 sft=17,034
MCQ regressions              (empty)
MCQ gains                    n=  2  pub= 41,442  sft= 33,945  Δ= -7,497 (-18.1%)  med pub=41,442 sft=33,945
MCQ both right               n=  6  pub= 15,409  sft= 15,929  Δ=   +520 ( +3.4%)  med pub=14,110 sft=12,564
MCQ both wrong               n=  1  pub= 47,146  sft= 24,602  Δ=-22,544 (-47.8%)  med pub=47,146 sft=24,602
FF all                       n= 12  pub= 11,798  sft=  7,824  Δ= -3,974 (-33.7%)  med pub=10,474 sft= 6,719
FF regressions               (empty)
FF gains                     (empty)
FF both right                n=  4  pub=  4,041  sft=  3,957  Δ=    -84 ( -2.1%)  med pub= 3,385 sft= 2,798
FF both wrong                n=  8  pub= 15,676  sft=  9,758  Δ= -5,918 (-37.8%)  med pub=12,993 sft= 7,987


In [14]:
# Format diagnostics — answer placement & multi-blank `[ANS]` discipline
#   H1 (MCQ): does sft-005 lose `\boxed{LETTER}` emission in final section on regressions?
#   H2 (FF):  does sft-005 lose multi-blank `\boxed{}` count on regressions (format collapse)?
#   H3 (MCQ): does sft-005 swap to the wrong letter while still emitting clean format (real reasoning loss)?
import re

BOXED        = re.compile(r'\\boxed\{([^}]*)\}')
MCQ_LETTER   = re.compile(r'\\boxed\{\s*([A-E])\s*\}')

def split_think(resp):
    if '</think>' in resp:
        idx = resp.index('</think>')
        return resp[:idx], resp[idx + len('</think>'):]
    return resp, ''

def final_section(resp):
    _, final = split_think(resp)
    return final if final.strip() else resp  # fallback if no </think>

def gold_count(r):
    """Expected number of answers: list length for multi-blank, else 1."""
    g = r.get('gold')
    if isinstance(g, list):
        return len(g)
    return 1 if g not in (None, '') else 0

def fmt_features(r):
    final = final_section(r['response'])
    boxed = BOXED.findall(final)
    return {
        'boxed_count_final': len(boxed),
        'boxed_first':       (boxed[0] if boxed else None),
        'gold_count':        gold_count(r),
        'len':               len(r['response']),
    }

def summarize(label, ids, rows_pub, rows_sft):
    if not ids:
        print(f"{label:<22} (empty)"); return
    pub_box = [fmt_features(rows_pub[i])['boxed_count_final'] for i in ids]
    sft_box = [fmt_features(rows_sft[i])['boxed_count_final'] for i in ids]
    pub_has = sum(1 for x in pub_box if x >= 1)
    sft_has = sum(1 for x in sft_box if x >= 1)
    pub_mean = sum(pub_box) / len(pub_box)
    sft_mean = sum(sft_box) / len(sft_box)
    print(f"{label:<22} n={len(ids):>3}  "
          f"final-\\boxed present: pub {pub_has}/{len(ids)} → sft {sft_has}/{len(ids)}   "
          f"mean count: pub {pub_mean:.2f} → sft {sft_mean:.2f}")

print("=== MCQ: `\\boxed{LETTER}` in final section ===")
for label, ids in [('MCQ all', mcq_ids), ('MCQ regressions', mcq_regr),
                   ('MCQ gains', mcq_gain), ('MCQ both right', mcq_RR), ('MCQ both wrong', mcq_WW)]:
    summarize(label, ids, pub002, sft005)

print("\n=== Free-form: `\\boxed{}` count in final section (multi-blank discipline) ===")
for label, ids in [('FF all', ff_ids), ('FF regressions', ff_regr),
                   ('FF gains', ff_gain), ('FF both right', ff_RR), ('FF both wrong', ff_WW)]:
    summarize(label, ids, pub002, sft005)

# H2b: For free-form regressions, did sft emit fewer \boxed answers than gold expects?
print("\n=== FF regressions: expected gold count vs emitted \\boxed in final ===")
fmt_collapse = 0
under_gold   = 0
for i in sorted(ff_regr):
    expected = gold_count(pub002[i])
    pub_box  = fmt_features(pub002[i])['boxed_count_final']
    sft_box  = fmt_features(sft005[i])['boxed_count_final']
    flags = []
    if sft_box < pub_box:    flags.append('sft<pub')
    if sft_box < expected:   flags.append('sft<gold')
    if sft_box < pub_box:    fmt_collapse += 1
    if sft_box < expected:   under_gold += 1
    flag_str = f"  <-- {','.join(flags)}" if flags else ''
    print(f"  id={i:>5}  expected={expected}  pub \\boxed={pub_box}  sft \\boxed={sft_box}{flag_str}")
if ff_regr:
    n = len(ff_regr)
    print(f"\nFF regressions: sft \\boxed-count < pub:  {fmt_collapse}/{n} ({fmt_collapse/n*100:.1f}%)")
    print(f"FF regressions: sft \\boxed-count < gold: {under_gold}/{n} ({under_gold/n*100:.1f}%)  "
          f"— high % ⇒ format collapse / answer dropping")

# H3: For MCQ regressions, did sft still emit a clean letter? If yes ⇒ real reasoning swap, not format loss.
print("\n=== MCQ regressions: clean-letter emission ===")
for i in sorted(mcq_regr):
    pub_final = final_section(pub002[i]['response'])
    sft_final = final_section(sft005[i]['response'])
    pub_m = MCQ_LETTER.search(pub_final)
    sft_m = MCQ_LETTER.search(sft_final)
    pub_letter = pub_m.group(1) if pub_m else '∅'
    sft_letter = sft_m.group(1) if sft_m else '∅'
    gold = pub002[i].get('gold', '?')
    diag = '(no letter)' if sft_letter == '∅' else ('(correct emit, swap)' if sft_letter != gold else '(matches gold??)')
    print(f"  id={i:>5}  gold={gold!s:<6}  pub→{pub_letter}  sft→{sft_letter}  {diag}")

=== MCQ: `\boxed{LETTER}` in final section ===
MCQ all                n=  9  final-\boxed present: pub 7/9 → sft 9/9   mean count: pub 1.00 → sft 1.22
MCQ regressions        (empty)
MCQ gains              n=  2  final-\boxed present: pub 0/2 → sft 2/2   mean count: pub 0.00 → sft 1.50
MCQ both right         n=  6  final-\boxed present: pub 6/6 → sft 6/6   mean count: pub 1.33 → sft 1.17
MCQ both wrong         n=  1  final-\boxed present: pub 1/1 → sft 1/1   mean count: pub 1.00 → sft 1.00

=== Free-form: `\boxed{}` count in final section (multi-blank discipline) ===
FF all                 n= 12  final-\boxed present: pub 11/12 → sft 12/12   mean count: pub 2.08 → sft 3.50
FF regressions         (empty)
FF gains               (empty)
FF both right          n=  4  final-\boxed present: pub 4/4 → sft 4/4   mean count: pub 1.25 → sft 1.25
FF both wrong          n=  8  final-\boxed present: pub 7/8 → sft 8/8   mean count: pub 2.50 → sft 4.62

=== FF regressions: expected gold count vs emitt

In [16]:
# Side-by-side trace printer — drill into a specific regression / gain id.
# Set QID to any id from the printed lists above; if None, picks first non-empty bucket.
QID  = None
SHOW = 'final'  # 'final' | 'think' | 'full'

def section(resp, which):
    think, final = split_think(resp)
    return {'think': think, 'final': final, 'full': resp}[which]

if QID is None:
    for label, ids in [('FF regr', ff_regr), ('MCQ regr', mcq_regr),
                       ('FF gain', ff_gain), ('MCQ gain', mcq_gain)]:
        if ids:
            QID = next(iter(ids))
            print(f"[auto-picked {label} id={QID}]")
            break
    else:
        QID = next(iter(shared), None)
        print(f"[no regressions/gains; falling back to shared id={QID}]")

if QID is None or QID not in pub002 or QID not in sft005:
    print(f"id={QID} missing or no shared ids")
else:
    p, s = pub002[QID], sft005[QID]
    print(f"=== id={QID}  is_mcq={p['is_mcq']}  gold={p.get('gold')!r} ===")
    print(f"pub-002: correct={p['correct']}  len={len(p['response']):,}")
    print(f"sft-005: correct={s['correct']}  len={len(s['response']):,}")
    print(f"\n----- pub-002 [{SHOW}] -----\n{section(p['response'], SHOW)}")
    print(f"\n----- sft-005 [{SHOW}] -----\n{section(s['response'], SHOW)}")

[auto-picked MCQ gain id=148]
=== id=148  is_mcq=True  gold='C' ===
pub-002: correct=False  len=39,801
sft-005: correct=True  len=31,120

----- pub-002 [final] -----


----- sft-005 [final] -----


\boxed{C}
